# Experiment: Founder Approval Record Gate

Objective: reduce founder approval state to public-safe labels without leaking private approval content or turning approval into contact permission.


In [ ]:
from __future__ import annotations

APPROVAL_NOT_RECORDED = "approval_not_recorded"
APPROVAL_DENIED = "approval_denied"
FOUNDER_REVIEW_ONLY = "founder_review_only"
SCOPED_APPROVAL = "approval_scope_public_safe_quote_only"
APPROVAL_RELEASE_BLOCKED = "approval_release_blocked"
EXPIRED = "approval_expired_or_superseded"

REQUIRED_SCOPED_FLAGS = {
    "approval_recorded",
    "founder_identity_private",
    "packet_version_label",
    "scope_public_safe_quote_only",
    "recipient_qualification_required",
    "response_capture_required",
    "blocked_content_acknowledged",
}

RELEASE_BLOCKERS = {
    "raw_message_in_public",
    "private_contact_in_public",
    "patient_records_in_public",
    "patient_samples_scope",
    "treatment_or_dosing_scope",
    "trial_screening_scope",
    "purchase_scope",
    "import_scope",
    "procurement_scope",
    "ambiguous_scope",
}


def classify_approval_record(record: dict[str, bool]) -> str:
    """Return the public-safe approval label for one record."""
    if any(record.get(flag, False) for flag in RELEASE_BLOCKERS):
        return APPROVAL_RELEASE_BLOCKED
    if record.get("approval_denied", False):
        return APPROVAL_DENIED
    if not record.get("approval_recorded", False):
        return APPROVAL_NOT_RECORDED
    if record.get("approval_expired_or_superseded", False):
        return EXPIRED
    if any(not record.get(flag, False) for flag in REQUIRED_SCOPED_FLAGS):
        return FOUNDER_REVIEW_ONLY
    return SCOPED_APPROVAL


def can_enter_private_send_workflow(record: dict[str, bool]) -> bool:
    """Return True only when approval, recipient, and packet gates pass."""
    if classify_approval_record(record) != SCOPED_APPROVAL:
        return False
    if not record.get("recipient_qualified", False):
        return False
    return record.get("outbound_packet_gate_passed", False)


base_scoped_record = {
    "approval_recorded": True,
    "founder_identity_private": True,
    "packet_version_label": True,
    "scope_public_safe_quote_only": True,
    "recipient_qualification_required": True,
    "response_capture_required": True,
    "blocked_content_acknowledged": True,
}

test_cases = {
    "no_record": {},
    "denied": {"approval_recorded": True, "approval_denied": True},
    "review_only": {"approval_recorded": True, "founder_identity_private": True},
    "scoped_public_safe": base_scoped_record,
    "raw_private_public": {**base_scoped_record, "raw_message_in_public": True},
    "procurement_scope": {**base_scoped_record, "procurement_scope": True},
}

expected = {
    "no_record": APPROVAL_NOT_RECORDED,
    "denied": APPROVAL_DENIED,
    "review_only": FOUNDER_REVIEW_ONLY,
    "scoped_public_safe": SCOPED_APPROVAL,
    "raw_private_public": APPROVAL_RELEASE_BLOCKED,
    "procurement_scope": APPROVAL_RELEASE_BLOCKED,
}

results = {
    name: classify_approval_record(record) for name, record in test_cases.items()
}
assert results == expected
assert not can_enter_private_send_workflow(base_scoped_record)
assert can_enter_private_send_workflow(
    {
        **base_scoped_record,
        "recipient_qualified": True,
        "outbound_packet_gate_passed": True,
    }
)

decision_summary = {
    "results": results,
    "send_boundary": "requires recipient qualification and outbound packet gate",
    "public_repo_boundary": "labels only; no raw approval messages or contacts",
}
decision_summary

## Decision

Continue with the approval-record gate. Do not send, select recipients, request samples, or interpret approval as medical action from this public notebook.
